# PyTorch Interview Cheat Sheet

> **30–60 minute review · 5-minute lookup · last-minute interview recall**

Priority: **🔥 Must Know** · **⭐ Useful** · **🟡 Interview Useful**  
Pattern: **syntax → meaning → tiny example → shape/result**

Use the table of contents or search for `CrossEntropy`, `DataLoader`, `reshape`, `attention`, `state_dict`, or `device`.

## Contents

1. Imports and Setup · 2. Tensor Creation · 3. Tensor Properties · 4. Dtypes · 5. Devices  
6. Indexing and Slicing · 7. Reshaping and Dimensions · 8. Combining and Splitting  
9. Reductions · 10. Broadcasting · 11. Masking · 12. Sorting and Top-K · 13. Matrix Multiplication  
14. Autograd · 15. `nn.Module` · 16. Common Layers · 17. Activations · 18. Normalization and Dropout  
19. Embeddings · 20. CNN Layers · 21. Loss Functions · 22. Optimizers · 23. Training Loop  
24. Validation / Inference · 25. Dataset and DataLoader · 26. GPU Handling · 27. Mixed Precision  
28. Debugging · 29. Saving and Loading · 30. Transformer / Attention · 31. Shape Reference  
32. Task → Output → Loss · 33. Common Mistakes · 34. Interview Quick Answers · 35. One-Page Memory


## 1. Imports and Setup — 🔥 Must Know

Set the seed for reproducible examples. CUDA is preferred, then Apple MPS, then CPU.


In [ ]:
import math

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.utils.data import (
    Dataset,
    TensorDataset,
    DataLoader,
    random_split,
)

torch.manual_seed(42)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif (
    hasattr(torch.backends, "mps")
    and torch.backends.mps.is_available()
):
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("PyTorch:", torch.__version__)
print("device:", device)


## 2. Tensor Creation — 🔥 Must Know

| Goal | Syntax |
|---|---|
| From values | `torch.tensor([1, 2, 3])` |
| Zeros / ones | `torch.zeros(2, 3)` / `torch.ones(2, 3)` |
| Constant | `torch.full((2, 3), 5)` |
| Range | `torch.arange(0, 10, 2)` |
| Even spacing | `torch.linspace(0, 1, 5)` |
| Uniform / normal | `torch.rand(2, 3)` / `torch.randn(2, 3)` |
| Random integers | `torch.randint(0, 10, (2, 3))` |
| Identity | `torch.eye(4)` |
| Match another | `torch.zeros_like(x)` / `torch.ones_like(x)` |


In [ ]:
values = torch.tensor([1, 2, 3])
x = torch.arange(6, dtype=torch.float32).reshape(2, 3)
examples = {
    "zeros": torch.zeros(2, 3),
    "ones": torch.ones(2, 3),
    "full": torch.full((2, 3), 5),
    "range": torch.arange(0, 10, 2),
    "linspace": torch.linspace(0, 1, 5),
    "rand": torch.rand(2, 3),
    "randn": torch.randn(2, 3),
    "randint": torch.randint(0, 10, (2, 3)),
    "eye": torch.eye(4),
    "zeros_like": torch.zeros_like(x),
}
print(values)
print({name: tuple(t.shape) for name, t in examples.items()})


## 3. Tensor Properties

| API | Meaning |
|---|---|
| `x.shape`, `x.size()` | dimensions (`torch.Size`) |
| `x.ndim` | number of dimensions |
| `x.numel()` | total elements |
| `x.dtype` | element type |
| `x.device` | storage device |
| `x.requires_grad` | whether Autograd tracks operations |


In [ ]:
x = torch.randn(32, 3, 224, 224)
print(x.shape, x.size())
print("ndim:", x.ndim, "numel:", x.numel())
print("dtype:", x.dtype, "device:", x.device, "grad:", x.requires_grad)


## 4. Dtypes — 🔥 Must Know

| Dtype | Typical use |
|---|---|
| `torch.float32` | default training/inference float |
| `torch.float64` | extra precision; uncommon in deep learning |
| `torch.float16` | mixed precision; limited numeric range |
| `torch.bfloat16` | mixed precision; float32-like exponent range |
| `torch.int64` / `torch.long` | class indices, token IDs, indexing |
| `torch.int32` | integer data/interoperability |
| `torch.bool` | masks |

Conversions: `x.float()`, `x.long()`, `x.int()`, `x.bool()`, `x.to(torch.float32)`.

> 🔥 `CrossEntropyLoss` class-index targets are usually `torch.long`.  
> 🔥 `BCEWithLogitsLoss` targets are usually floating point.


In [ ]:
x = torch.tensor([0, 1, 2])
print(x.dtype, x.float().dtype, x.bool().dtype)


## 5. Devices — 🔥 Must Know

```python
x = x.to(device)
model = model.to(device)

print(x.device)
print(next(model.parameters()).device)

x = x.cpu()
array = x.detach().cpu().numpy()  # safe NumPy conversion
```

**Invariant:** model device = input device = target device for normal training operations.  
`x.to(device)` is not in-place—assign or use the returned tensor.


## 6. Indexing and Slicing

Standard Python indexing rules apply. Tensor boolean conditions use `&`, `|`, `~`—not Python `and`, `or`, `not`.


In [ ]:
x = torch.tensor([
    [10, 20, 30],
    [40, 50, 60],
    [70, 80, 90],
])

print("row:", x[0])
print("column:", x[:, 0])
print("scalar:", x[1, 2])
print("rows 1+:", x[1:])
print("cols 1+:", x[:, 1:])
print("last:", x[-1])
print("> 50:", x[x > 50])
print("20 < x < 80:", x[(x > 20) & (x < 80)])


## 7. Reshaping and Dimensions — 🔥 Must Know

| API | Rapid meaning |
|---|---|
| `reshape(...)` | flexible reshape; may view or copy |
| `view(...)` | reshape requiring compatible memory layout |
| `flatten(start_dim=...)` | collapse dimensions |
| `unsqueeze(dim)` | insert size-1 dimension |
| `squeeze(dim)` | remove a size-1 dimension |
| `transpose(a, b)` | swap two dimensions |
| `permute(...)` | reorder all dimensions |

> Use `reshape` when only the result shape matters; use `view` when view-like semantics matter and layout supports it.  
> ⚠️ Bare `squeeze()` removes **every** dimension of size 1.


In [ ]:
images = torch.randn(32, 3, 28, 28)
print("reshape:", images.reshape(32, -1).shape)  # (32, 2352)
print("flatten:", images.flatten(1).shape)       # (32, 2352)

x = torch.randn(3, 4)
print("unsqueeze:", x.unsqueeze(0).shape, x.unsqueeze(1).shape)
x = torch.randn(1, 3, 1, 4)
print("squeeze:", x.squeeze().shape)

images = torch.randn(32, 3, 224, 224)
print("NCHW -> NHWC:", images.permute(0, 2, 3, 1).shape)
print("transpose C/H:", images.transpose(1, 2).shape)


## 8. Combining and Splitting Tensors — 🔥 Must Know

**`cat` joins along an existing dimension; `stack` creates a new dimension.**

| Expression for `a,b: (3,4)` | Result |
|---|---|
| `torch.cat([a,b], dim=0)` | `(6,4)` |
| `torch.cat([a,b], dim=1)` | `(3,8)` |
| `torch.stack([a,b], dim=0)` | `(2,3,4)` |

Splitting: `torch.chunk(x, chunks=4, dim=0)` specifies a number of chunks; `torch.split(x, split_size_or_sections=..., dim=0)` specifies chunk size or exact sizes.


In [ ]:
a = torch.randn(3, 4)
b = torch.randn(3, 4)
print(torch.cat([a, b], dim=0).shape)
print(torch.cat([a, b], dim=1).shape)
print(torch.stack([a, b], dim=0).shape)
print([part.shape for part in torch.chunk(torch.arange(12), chunks=4)])
print([part.shape for part in torch.split(torch.arange(10), [3, 2, 5])])


## 9. Reductions — 🔥 `dim` Is Interview Essential

APIs: `sum`, `mean`, `max`, `min`, `argmax`, `argmin`, `torch.any`, `torch.all`.

Reduction over `dim=k` removes dimension `k`, unless `keepdim=True`.

| Input `(2,3)` | Result |
|---|---|
| `x.sum(dim=0)` | `(3,)` |
| `x.sum(dim=1)` | `(2,)` |
| `x.sum(dim=1, keepdim=True)` | `(2,1)` |
| `x.argmax(dim=1)` | one index per row: `(2,)` |


In [ ]:
x = torch.tensor([[1.0, 5.0, 2.0], [4.0, 3.0, 6.0]])
print(x.sum(), x.mean(), x.max(), x.min())
print(x.sum(dim=0).shape, x.sum(dim=1).shape)
print(x.sum(dim=1, keepdim=True).shape, x.argmax(dim=1))
print(torch.any(x > 5), torch.all(x > 0))


## 10. Broadcasting — 🔥 Interview Essential

Compare dimensions **from the right**. Dimensions are compatible if equal or one is `1`.

| Operation | Result |
|---|---|
| `(32,10) + (10,)` | `(32,10)` |
| `(5,1) + (1,7)` | `(5,7)` |
| `(3,4) + (3,)` | ❌ incompatible |

Feature normalization preserves the broadcastable feature axis with `keepdim=True`:


In [ ]:
x = torch.randn(32, 10)
mean = x.mean(dim=0, keepdim=True)
std = x.std(dim=0, keepdim=True)
x_normalized = (x - mean) / (std + 1e-8)
print(mean.shape, std.shape, x_normalized.shape)


## 11. Masking and Conditional Operations

```python
mask = x > 0
positive = x[mask]                     # selected values; flattened
y = torch.where(condition, if_true, if_false)
y = torch.clamp(x, min=0)              # simple clipping
```


In [ ]:
x = torch.tensor([-2.0, 0.5, 3.0])
positive = x[x > 0]
via_where = torch.where(x < 0, torch.tensor(0.0), x)
print(positive, via_where, torch.clamp(x, min=0))


## 12. Sorting and Top-K

```python
values, indices = torch.sort(x, dim=-1)     # values + indices
indices = torch.argsort(x, dim=-1)          # indices only
top_values, top_indices = torch.topk(x, k=5, dim=-1)
```


In [ ]:
x = torch.tensor([[0.3, 0.9, 0.1, 0.7, 0.5, 0.2]])
values, indices = torch.sort(x, dim=-1)
top_values, top_indices = torch.topk(x, k=3, dim=-1)
print(values, indices)
print(top_values, top_indices)


## 13. Matrix Multiplication and Advanced Selection

`*` is element-wise; `@` is matrix multiplication.

| API | Use |
|---|---|
| `a @ b`, `torch.matmul(a,b)` | matrix/batched matrix multiply with broadcasting |
| `torch.mm(a,b)` | 2-D matrix multiply |
| `torch.bmm(a,b)` | 3-D batch matrix multiply; no broadcasting |

`(32,128) @ (128,64) → (32,64)`  
`bmm((16,20,64), (16,64,32)) → (16,20,32)`

Class-score selection uses paired row/class indices. `gather` generalizes indexed selection along a dimension.


In [ ]:
a = torch.randn(32, 128)
b = torch.randn(128, 64)
print("matmul:", (a @ b).shape)
A = torch.randn(16, 20, 64)
B = torch.randn(16, 64, 32)
print("bmm:", torch.bmm(A, B).shape)

scores = torch.randn(32, 10)
labels = torch.randint(0, 10, (32,))
selected = scores[torch.arange(scores.size(0)), labels]
gathered = torch.gather(scores, dim=1, index=labels.unsqueeze(1)).squeeze(1)
print("selected:", selected.shape, "gather matches:", torch.equal(selected, gathered))


## 14. Autograd — 🔥 Must Know

**Forward builds the computation graph; backward computes gradients.**

Core APIs: `requires_grad=True`, `loss.backward()`, `x.grad`, `x.grad.zero_()`, `x.detach()`, `x.requires_grad_(True)`, `torch.no_grad()`.

PyTorch gradients **accumulate**. Memorize: `zero_grad → forward → loss → backward → step`.

| API | Meaning |
|---|---|
| `x.detach()` | disconnect this tensor from the graph |
| `with torch.no_grad():` | disable gradient recording for a block |
| `with torch.inference_mode():` | stronger inference-only optimization |


In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2
y.backward()
print("dy/dx:", x.grad)  # 4

x.grad.zero_()
(3 * x).backward()
print("new grad after clearing:", x.grad)
print("detached tracks grad:", x.detach().requires_grad)


## 15. `nn.Module` — 🔥 Must Know

`__init__` defines layers; `forward` defines computation; `super().__init__()` initializes module machinery. Call `model(x)` (hooks and framework behavior), not normally `model.forward(x)`.


In [ ]:
class Model(nn.Module):
    def __init__(self, in_features=10, num_classes=3):
        super().__init__()
        self.fc = nn.Linear(in_features, num_classes)

    def forward(self, x):
        return self.fc(x)


model = Model()
print(model(torch.randn(4, 10)).shape)  # (B, 3)


## 16. Common Layers

| Layer | Purpose / shape idea |
|---|---|
| `nn.Linear(a,b)` | last dim `a → b`; weight `(b,a)`, bias `(b,)` |
| `nn.ReLU()` | common MLP/CNN hidden activation |
| `nn.GELU()` | common Transformer activation |
| `nn.Dropout(p)` | random regularization during training |
| `nn.BatchNorm1d(n)` | batch normalization |
| `nn.LayerNorm(n)` | normalize last feature dimension |
| `nn.Flatten()` | flatten feature dimensions |
| `nn.Embedding(v,d)` | integer IDs → vectors |
| `nn.Conv2d(...)` | image convolution |
| `nn.MaxPool2d(...)` | spatial downsampling |

For `nn.Linear(128,64)`: input `(B,128)` → `(B,64)`; parameters = `128×64 + 64`.


In [ ]:
layer = nn.Linear(128, 64)
print(layer(torch.randn(8, 128)).shape)
print(layer.weight.shape, layer.bias.shape)
print("parameters:", sum(p.numel() for p in layer.parameters()))


## 17. Activation Functions and Logits

| Activation | Typical use |
|---|---|
| `nn.ReLU()` / `nn.LeakyReLU()` | MLP/CNN hidden layers |
| `nn.GELU()` | Transformer hidden layers |
| `nn.Sigmoid()` | convert binary logits to probabilities |
| `nn.Tanh()` | bounded `[-1,1]` output/state |
| `nn.Softmax(dim=...)` | convert multiclass logits to probabilities |

**Logits = raw scores before sigmoid/softmax.** Binary `(B,1)`, multiclass `(B,C)`, language model `(B,T,V)`.

> 🔥 Do **not** softmax before `CrossEntropyLoss`.  
> 🔥 Do **not** sigmoid before `BCEWithLogitsLoss`.


## 18. Normalization and Dropout

| Feature | Training | Evaluation |
|---|---|---|
| Dropout | randomly drops activations | disabled |
| BatchNorm | batch statistics + running-stat updates | stored running statistics |
| LayerNorm | per sample/token features | same computation |

`model.train()` activates training behavior. `model.eval()` activates evaluation behavior.  
**BatchNorm:** batch-dependent, common in CNNs. **LayerNorm:** independent of batch statistics, common in Transformers.


In [ ]:
regularized = nn.Sequential(
    nn.Linear(16, 16),
    nn.BatchNorm1d(16),
    nn.ReLU(),
    nn.Dropout(0.5),
)
transformer_norm = nn.LayerNorm(16)
print(regularized(torch.randn(4, 16)).shape)
print(transformer_norm(torch.randn(4, 7, 16)).shape)


## 19. Embeddings

```python
embedding = nn.Embedding(vocab_size, hidden_size)
```

Input token IDs: `(B,T)`, dtype `torch.long` → output embeddings `(B,T,D)`.  
Example: `(8,128) → Embedding(...,768) → (8,128,768)`.


In [ ]:
embedding = nn.Embedding(1000, 64)
token_ids = torch.randint(0, 1000, (8, 128), dtype=torch.long)
print(embedding(token_ids).shape)


## 20. CNN Layers

PyTorch image convention: **NCHW** = `(batch, channels, height, width)`.

```python
nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)
nn.MaxPool2d(2)
```

Kernel 3, padding 1, stride 1 preserves spatial size. Pool 2 typically halves it: `(B,32,64,64) → (B,32,32,32)`.

Conv parameter count = `out_channels × in_channels × kernel_h × kernel_w + out_channels` (with bias).


In [ ]:
conv = nn.Conv2d(3, 32, kernel_size=3, padding=1)
pool = nn.MaxPool2d(2)
images = torch.randn(4, 3, 64, 64)
features = conv(images)
print(features.shape, pool(features).shape)
print("conv parameters:", sum(p.numel() for p in conv.parameters()))


## 21. Loss Functions — 🔥 Memorize

| Task | Model output | Target | Loss |
|---|---|---|---|
| Regression | continuous `(B,1)` | float | `nn.MSELoss()` / `nn.L1Loss()` |
| Binary | logits `(B,1)` | float `(B,1)` | `nn.BCEWithLogitsLoss()` |
| Multiclass | logits `(B,C)` | long `(B,)` | `nn.CrossEntropyLoss()` |
| Multilabel | logits `(B,C)` | float `(B,C)` | `nn.BCEWithLogitsLoss()` |
| Language model | logits `(B,T,V)` | long `(B,T)` | `nn.CrossEntropyLoss()` |

**MSE:** squares errors, more sensitive to outliers. **L1:** absolute errors.


In [ ]:
# Binary
binary_logits = torch.randn(8, 1)
binary_targets = torch.randint(0, 2, (8, 1)).float()
binary_loss = nn.BCEWithLogitsLoss()(binary_logits, binary_targets)
binary_probs = torch.sigmoid(binary_logits)
binary_pred = (binary_probs >= 0.5).long()

# Multiclass
class_logits = torch.randn(8, 5)
class_targets = torch.randint(0, 5, (8,), dtype=torch.long)
class_loss = nn.CrossEntropyLoss()(class_logits, class_targets)
class_pred = class_logits.argmax(dim=1)
accuracy = (class_pred == class_targets).float().mean()
print(binary_loss.item(), class_loss.item(), accuracy.item())


## 22. Optimizers

```python
optim.SGD(model.parameters(), lr=0.01)
optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
optim.Adam(model.parameters(), lr=1e-3)
optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
```

| Optimizer | Recall |
|---|---|
| SGD | simple baseline |
| SGD + momentum | smoother momentum-based updates |
| Adam | adaptive per-parameter updates |
| AdamW | Adam + decoupled weight decay; common modern default |


## 23. Standard Training Loop — 🔥 Write From Memory

```python
model.train()
for X, y in train_loader:
    X = X.to(device)
    y = y.to(device)
    optimizer.zero_grad()
    logits = model(X)
    loss = loss_fn(logits, y)
    loss.backward()
    optimizer.step()
```

For dataset-level metrics, weight mean batch loss by batch size:

```python
total_loss += loss.item() * y.size(0)
total_correct += (logits.argmax(dim=1) == y).sum().item()
total_samples += y.size(0)
avg_loss = total_loss / total_samples
accuracy = total_correct / total_samples
```


In [ ]:
# Tiny, fully executable training example
X_all = torch.randn(96, 10)
y_all = torch.randint(0, 3, (96,))
train_dataset = TensorDataset(X_all, y_all)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

model = Model().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)

model.train()
total_loss = total_correct = total_samples = 0
for X, y in train_loader:
    X, y = X.to(device), y.to(device)
    optimizer.zero_grad()
    logits = model(X)
    loss = loss_fn(logits, y)
    loss.backward()
    optimizer.step()
    total_loss += loss.item() * y.size(0)
    total_correct += (logits.argmax(dim=1) == y).sum().item()
    total_samples += y.size(0)

print("loss:", total_loss / total_samples, "accuracy:", total_correct / total_samples)


## 24. Validation / Inference Loop

```python
model.eval()
with torch.inference_mode():
    for X, y in val_loader:
        X = X.to(device)
        y = y.to(device)
        logits = model(X)
        loss = loss_fn(logits, y)
        # NO backward(); NO optimizer.step()
```

| API | Role |
|---|---|
| `model.train()` | training behavior for Dropout/BatchNorm |
| `model.eval()` | evaluation behavior for Dropout/BatchNorm |
| `torch.inference_mode()` | disable Autograd overhead |

🔥 `model.eval()` **does not** disable gradients. Usually use `eval()` and `inference_mode()` together.


In [ ]:
model.eval()
val_loss = val_correct = val_samples = 0
with torch.inference_mode():
    for X, y in DataLoader(train_dataset, batch_size=64, shuffle=False):
        X, y = X.to(device), y.to(device)
        logits = model(X)
        loss = loss_fn(logits, y)
        val_loss += loss.item() * y.size(0)
        val_correct += (logits.argmax(dim=1) == y).sum().item()
        val_samples += y.size(0)
print("val loss:", val_loss / val_samples, "val accuracy:", val_correct / val_samples)


## 25. Dataset and DataLoader

**Dataset → one sample. DataLoader → batching, shuffling, workers, iteration.**

```python
class MyDataset(Dataset):
    def __init__(self, X, y):
        self.X, self.y = X, y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

dataset = TensorDataset(X, y)  # aligned tensors already in memory
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
```

| Option | Meaning |
|---|---|
| `shuffle=True` | randomize sample order each iteration |
| `num_workers=4` | load using worker processes |
| `pin_memory=True` | may accelerate CPU→CUDA copies |
| `drop_last=True` | discard partial final batch |
| `collate_fn=...` | custom sample-to-batch logic |

Split: `train_dataset, val_dataset = random_split(dataset, [0.8, 0.2], generator=...)` (fractional lengths supported by modern PyTorch).


In [ ]:
class MyDataset(Dataset):
    def __init__(self, X, y):
        self.X, self.y = X, y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


dataset = MyDataset(torch.randn(20, 4), torch.randint(0, 2, (20,)))
train_part, val_part = random_split(
    dataset, [16, 4], generator=torch.Generator().manual_seed(42)
)
loader = DataLoader(train_part, batch_size=8, shuffle=True)
batch_X, batch_y = next(iter(loader))
print(len(dataset), batch_X.shape, batch_y.shape)


### Variable-Length Text — ⭐ Useful

```python
from torch.nn.utils.rnn import pad_sequence

padded = pad_sequence(sequences, batch_first=True, padding_value=0)
attention_mask = padded != 0
```

Tokens `(B,T)`; mask `(B,T)`. In real tokenizers, derive masks from the configured padding ID.


In [ ]:
from torch.nn.utils.rnn import pad_sequence

sequences = [torch.tensor([4, 2, 7]), torch.tensor([9, 3])]
padded = pad_sequence(sequences, batch_first=True, padding_value=0)
attention_mask = padded != 0
print(padded)
print(attention_mask)


## 26. GPU and Device Handling

Device-aware transfer:

```python
for X, y in loader:
    use_non_blocking = device.type == "cuda"
    X = X.to(device, non_blocking=use_non_blocking)
    y = y.to(device, non_blocking=use_non_blocking)
```

`pin_memory=True` + `non_blocking=True` can improve CUDA transfers, but improvement is workload-dependent.

CUDA memory APIs: `torch.cuda.memory_allocated()`, `memory_reserved()`, `max_memory_allocated()`.

Memory = parameters + gradients + optimizer state + activations + input batch + temporary buffers.

**OOM checklist:** reduce batch size · mixed precision · gradient accumulation · shorten sequence/image · check graph retention · avoid retaining GPU tensors.


In [ ]:
if torch.cuda.is_available():
    print("allocated:", torch.cuda.memory_allocated())
    print("reserved:", torch.cuda.memory_reserved())
    print("peak allocated:", torch.cuda.max_memory_allocated())
else:
    print("CUDA memory metrics skipped on", device.type)


## 27. Mixed Precision, Clipping, and Accumulation

### Mixed precision — ⭐ Useful

Modern CUDA pattern:

```python
scaler = torch.amp.GradScaler("cuda", enabled=True)

optimizer.zero_grad()
with torch.autocast(device_type="cuda", dtype=torch.float16):
    logits = model(X)
    loss = loss_fn(logits, y)

scaler.scale(loss).backward()
scaler.unscale_(optimizer)  # before clipping
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
scaler.step(optimizer)
scaler.update()
```

**FP16:** smaller range. **BF16:** larger exponent range; common on modern accelerators. Device capabilities vary.

### Gradient clipping

Order: `backward → clip → step`. With scaled gradients, unscale before clipping.

### Gradient accumulation

```python
accum_steps = 4
optimizer.zero_grad()
for step, (X, y) in enumerate(train_loader):
    loss = loss_fn(model(X), y) / accum_steps
    loss.backward()
    if (step + 1) % accum_steps == 0:
        optimizer.step()
        optimizer.zero_grad()
```

Physical batch × accumulation steps ≈ effective batch size. Handle a final incomplete accumulation window in production code.


In [ ]:
# Executable AMP smoke test only when CUDA supports the shown pattern.
if device.type == "cuda":
    amp_model = Model().to(device)
    amp_optimizer = optim.AdamW(amp_model.parameters(), lr=1e-3)
    amp_X = torch.randn(8, 10, device=device)
    amp_y = torch.randint(0, 3, (8,), device=device)
    scaler = torch.amp.GradScaler("cuda", enabled=True)
    amp_optimizer.zero_grad()
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        amp_loss = nn.CrossEntropyLoss()(amp_model(amp_X), amp_y)
    scaler.scale(amp_loss).backward()
    scaler.unscale_(amp_optimizer)
    torch.nn.utils.clip_grad_norm_(amp_model.parameters(), max_norm=1.0)
    scaler.step(amp_optimizer)
    scaler.update()
    print("CUDA AMP step complete")
else:
    print("CUDA AMP example skipped on", device.type)


## 28. Debugging — 🔥 Memorize

### First checklist

1. shape
2. dtype
3. device
4. target format
5. loss choice
6. finite values
7. gradients
8. parameter updates
9. train/eval mode
10. learning rate

```python
print(x.shape, x.dtype, x.device, x.requires_grad)
torch.isnan(x).any(); torch.isinf(x).any(); torch.isfinite(x).all()
```

Gradient interpretation: `grad is None` → no path, frozen, detached, or unused; huge → possible instability; near-zero → possible weak flow. There are no universal thresholds.


In [ ]:
for name, param in model.named_parameters():
    if param.grad is None:
        print(name, "NO GRAD")
    else:
        print(name, "grad norm:", param.grad.norm().item())

debug_x = torch.tensor([1.0, float("nan")])
print("finite:", torch.isfinite(debug_x).all().item())


### Anomaly Detection and Memory Leaks

```python
torch.autograd.set_detect_anomaly(True)
```

Useful for backward errors/non-finite gradients; slow—disable for normal training.

```python
# ❌ Keeps computation graphs alive
loss_history.append(loss)

# ✅ Store a Python scalar
loss_history.append(loss.item())

# ✅ Retain outputs without graphs or GPU storage
saved_outputs.append(output.detach().cpu())
```


## 29. Model Saving and Loading — 🔥 Interview Essential

Prefer `state_dict`: portable parameter names/tensors, but you must reconstruct the model architecture.


In [ ]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp_dir:
    model_path = Path(tmp_dir) / "model.pt"
    checkpoint_path = Path(tmp_dir) / "checkpoint.pt"

    # Weights only
    torch.save(model.state_dict(), model_path)
    restored_model = Model()
    state = torch.load(model_path, map_location=device, weights_only=True)
    restored_model.load_state_dict(state)
    restored_model.to(device).eval()

    # Training checkpoint
    checkpoint = {
        "epoch": 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_val_loss": val_loss / val_samples,
    }
    torch.save(checkpoint, checkpoint_path)
    loaded = torch.load(checkpoint_path, map_location=device, weights_only=False)
    restored_model.load_state_dict(loaded["model_state_dict"])
    optimizer.load_state_dict(loaded["optimizer_state_dict"])
    print("restored epoch:", loaded["epoch"])


Reference patterns:

```python
torch.save(model.state_dict(), "model.pt")

model = Model(...)
state = torch.load("model.pt", map_location=device, weights_only=True)
model.load_state_dict(state)
model.to(device)
model.eval()
```

Checkpoint:

```python
checkpoint = {
    "epoch": epoch,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "best_val_loss": best_val_loss,
}
torch.save(checkpoint, "checkpoint.pt")

checkpoint = torch.load("checkpoint.pt", map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
```


## 30. Transformer / Attention Syntax — 🔥 Memorize Shapes

`B` batch · `T` sequence · `D` hidden · `H` heads · `Dh = D/H` · `V` vocabulary

| Representation | Shape |
|---|---|
| Token IDs | `(B,T)` |
| Embeddings / Q / K / V | `(B,T,D)` |
| Split heads | `(B,H,T,Dh)` |
| Attention scores / weights | `(B,H,T,T)` |
| Per-head output | `(B,H,T,Dh)` |
| Combined output | `(B,T,D)` |
| LM logits | `(B,T,V)` |

`D % H == 0`; e.g. `768 / 12 = 64` per head.


In [ ]:
# Scaled dot-product attention with explicit heads
B, T, D, H = 2, 5, 16, 4
head_dim = D // H
x = torch.randn(B, T, D)
q_proj, k_proj, v_proj = nn.Linear(D, D), nn.Linear(D, D), nn.Linear(D, D)

def split_heads(t):
    return t.reshape(B, T, H, head_dim).transpose(1, 2)

Q = split_heads(q_proj(x))
K = split_heads(k_proj(x))
V = split_heads(v_proj(x))
scores = (Q @ K.transpose(-2, -1)) / math.sqrt(head_dim)
weights = torch.softmax(scores, dim=-1)
head_output = weights @ V
combined = head_output.transpose(1, 2).reshape(B, T, D)
print(Q.shape, scores.shape, head_output.shape, combined.shape)


### Q/K/V Mental Model and Causal Mask

`QKᵀ → scores`; scale by `sqrt(Dh)` for stable magnitudes; softmax → weights; `weights × V → output`.

```python
Q = q_proj(x); K = k_proj(x); V = v_proj(x)
scores = (Q @ K.transpose(-2, -1)) / math.sqrt(head_dim)
weights = torch.softmax(scores, dim=-1)
output = weights @ V
```

Causal masking prevents access to future positions:


In [ ]:
mask = torch.triu(
    torch.ones(T, T, dtype=torch.bool, device=scores.device),
    diagonal=1,
)
causal_scores = scores.masked_fill(mask, float("-inf"))
causal_weights = torch.softmax(causal_scores, dim=-1)
print(mask)
print("future weights are zero:", torch.all(causal_weights.masked_select(mask) == 0).item())


### `nn.MultiheadAttention`, FFN, Residuals

```python
attention = nn.MultiheadAttention(embed_dim=768, num_heads=12, batch_first=True)
output, weights = attention(x, x, x)  # self-attention

ffn = nn.Sequential(
    nn.Linear(D, 4 * D),
    nn.GELU(),
    nn.Linear(4 * D, D),
)  # (B,T,D) -> (B,T,D)

# Post-norm
x = norm1(x + attention_output)
x = norm2(x + ffn(x))

# Pre-norm variant
x = x + attention(norm1(x), norm1(x), norm1(x))[0]
```


In [ ]:
attention = nn.MultiheadAttention(embed_dim=D, num_heads=H, batch_first=True)
attention_output, attention_weights = attention(x, x, x)
ffn = nn.Sequential(nn.Linear(D, 4 * D), nn.GELU(), nn.Linear(4 * D, D))
norm1, norm2 = nn.LayerNorm(D), nn.LayerNorm(D)
x_block = norm1(x + attention_output)
x_block = norm2(x_block + ffn(x_block))
print(attention_output.shape, attention_weights.shape, x_block.shape)


### Language Model Loss and Next Token

`(B,T,V)` logits + `(B,T)` long targets become `B×T` independent vocabulary classification examples.

```python
loss = loss_fn(logits.reshape(-1, V), targets.reshape(-1))

next_logits = logits[:, -1, :]
greedy_token = next_logits.argmax(dim=-1)

probs = torch.softmax(next_logits, dim=-1)
sampled_token = torch.multinomial(probs, num_samples=1)
```

**KV cache:** cached keys and values from prior tokens, avoiding recomputation during autoregressive decoding.


In [ ]:
B, T, V = 2, 6, 100
lm_logits = torch.randn(B, T, V)
lm_targets = torch.randint(0, V, (B, T), dtype=torch.long)
lm_loss = nn.CrossEntropyLoss()(lm_logits.reshape(-1, V), lm_targets.reshape(-1))
next_logits = lm_logits[:, -1, :]
greedy_token = next_logits.argmax(dim=-1)
sampled_token = torch.multinomial(torch.softmax(next_logits, dim=-1), num_samples=1)
print(lm_loss.item(), greedy_token.shape, sampled_token.shape)


## 31. Common Shape Reference

| Data | Shape |
|---|---|
| Tabular | `(B,F)` |
| Binary output | `(B,1)` |
| Multiclass output | `(B,C)` |
| Image (PyTorch) | `(B,C,H,W)` |
| Token IDs | `(B,T)` |
| Embeddings | `(B,T,D)` |
| Attention | `(B,H,T,T)` |
| LM logits | `(B,T,V)` |


## 32. Task → Output → Loss — 🔥 Highest Priority Table

| Task | Output | Target | Loss | Prediction |
|---|---|---|---|---|
| Regression | `(B,1)` | float | MSE / L1 | raw value |
| Binary | `(B,1)` | float `(B,1)` | BCEWithLogits | sigmoid + threshold |
| Multiclass | `(B,C)` | long `(B,)` | CrossEntropy | `argmax(dim=1)` |
| Multilabel | `(B,C)` | float `(B,C)` | BCEWithLogits | sigmoid + per-label threshold |
| Language model | `(B,T,V)` | long `(B,T)` | CrossEntropy | next-token selection |


## 33. Common Mistakes — Wrong → Correct

| ❌ Wrong | ✅ Correct |
|---|---|
| Softmax before `CrossEntropyLoss` | pass raw logits |
| Sigmoid before `BCEWithLogitsLoss` | pass raw logits |
| CrossEntropy targets are float/one-hot | use `long` class indices for standard CE |
| BCE targets are integers | use floating targets matching logits shape |
| Model on GPU, batch on CPU | move model, inputs, and targets together |
| `x.to(device)` without assignment | `x = x.to(device)` |
| `optimizer.step()` before `backward()` | backward, then step |
| `zero_grad()` after backward but before step | zero before forward/backward |
| Forget gradients accumulate | clear them for independent steps |
| Validation uses `model.train()` | call `model.eval()` |
| Assume `eval()` disables gradients | also use `inference_mode()` / `no_grad()` |
| Store `loss` tensor every iteration | store `loss.item()` |
| Save GPU outputs with graph | `output.detach().cpu()` |
| Conv2d input is NHWC | PyTorch Conv2d expects NCHW |
| Wrong multiclass `argmax` dimension | usually `argmax(dim=1)` for `(B,C)` |
| Bare `squeeze()` unexpectedly removes batch | specify the intended dimension |
| Use `view` after non-contiguous `permute` | use `reshape` or `.contiguous().view(...)` |
| Confuse `cat` and `stack` | cat existing dim; stack new dim |
| Mean batch losses without weighting | weight by batch size for unequal final batch |
| Call `model.forward(x)` directly | call `model(x)` |
| Mask with Python `and` / `or` | use parenthesized `&` / `|` / `~` |
| Clip before gradients exist | clip after backward, before step |
| Clip scaled AMP gradients | unscale first, then clip |
| Load weights but forget `eval()` | call `.eval()` for inference |


## 34. Interview Quick Answers — 🟡 Interview Useful

1. **What is Autograd?** PyTorch's automatic differentiation system; it records tensor operations and computes gradients on backward.
2. **Why `zero_grad()`?** Gradients accumulate by default, so clear old gradients before an independent step.
3. **Why no softmax before CrossEntropy?** It expects raw logits and performs a numerically stable log-softmax-based calculation internally.
4. **Why no sigmoid before BCEWithLogits?** It combines sigmoid and binary cross-entropy stably.
5. **What is a logit?** A raw model score before probability conversion.
6. **`eval()` vs `no_grad()`?** `eval()` changes layer behavior; `no_grad()` disables gradient tracking.
7. **`inference_mode()` vs `no_grad()`?** Both disable gradients; inference mode removes more Autograd overhead and is stricter.
8. **Dataset vs DataLoader?** Dataset retrieves one sample; DataLoader batches, shuffles, parallelizes, and iterates.
9. **`reshape` vs `view`?** Reshape is layout-flexible; view requires compatible memory strides.
10. **`cat` vs `stack`?** Cat joins an existing dimension; stack creates a new one.
11. **What does reduction `dim` mean?** The axis collapsed by the reduction.
12. **How does broadcasting work?** Compare rightmost dimensions; sizes must match or one must be 1.
13. **Adam vs AdamW?** AdamW decouples weight decay from Adam's adaptive update.
14. **Why `model(x)`?** It invokes Module call machinery, including registered hooks.
15. **Why `state_dict`?** It is the portable mapping of parameter/buffer names to tensors.
16. **BatchNorm vs LayerNorm?** BatchNorm uses batch statistics; LayerNorm normalizes per sample/token features.
17. **What does Dropout do at evaluation?** Nothing; `eval()` disables random dropping.
18. **Why embeddings need long IDs?** IDs are integer lookup indices, not continuous input features.
19. **What is self-attention?** Queries, keys, and values come from the same sequence.
20. **Why divide attention by `sqrt(d_k)`?** To control score magnitude before softmax.
21. **Why multiple attention heads?** They provide parallel learned projections/subspaces for attending.
22. **What is a causal mask?** A mask blocking attention to future positions.
23. **What is a KV cache?** Stored prior keys/values used to avoid recomputation during autoregressive decoding.
24. **What causes GPU OOM besides batch size?** Activations, optimizer state, retained graphs, long sequences, and saved GPU tensors.
25. **Why gradient clipping?** To bound gradient magnitude and reduce instability from exploding gradients.
26. **What is effective batch with accumulation?** Roughly physical batch × accumulation steps.
27. **Why use `detach().cpu().numpy()`?** Detach from Autograd and move to CPU before NumPy conversion.


## 35. Final One-Page Memory Section — 🔥 Write This From Memory

### Device + Model

```python
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 3)
        )

    def forward(self, x):
        return self.net(x)
```

### Training + Validation

```python
model.train()
for X, y in train_loader:
    X, y = X.to(device), y.to(device)
    optimizer.zero_grad()
    logits = model(X)
    loss = loss_fn(logits, y)
    loss.backward()
    optimizer.step()

model.eval()
with torch.inference_mode():
    for X, y in val_loader:
        X, y = X.to(device), y.to(device)
        logits = model(X)
```

### Dataset + Classification

```python
class MyDataset(Dataset):
    def __init__(self, X, y): self.X, self.y = X, y
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

multiclass_loss = nn.CrossEntropyLoss()
class_pred = logits.argmax(dim=1)

binary_loss = nn.BCEWithLogitsLoss()
binary_pred = (torch.sigmoid(logits) >= 0.5).long()
```

### Attention + Save / Load

```python
scores = (Q @ K.transpose(-2, -1)) / math.sqrt(head_dim)
weights = torch.softmax(scores, dim=-1)
output = weights @ V

torch.save(model.state_dict(), path)
state = torch.load(path, map_location=device, weights_only=True)
model.load_state_dict(state)
```


### Final Memory Checklist

I should know these without documentation:

- [ ] tensor creation; shape / dtype / device inspection
- [ ] indexing; boolean masks; `&`, `|`, `~`
- [ ] reshape / flatten / squeeze / unsqueeze / transpose / permute
- [ ] cat vs stack; reductions and `dim`; broadcasting; matrix multiplication
- [ ] Autograd; gradient accumulation; `nn.Module`
- [ ] Linear / ReLU / GELU / Dropout; BatchNorm vs LayerNorm
- [ ] Embedding; Conv2d basics
- [ ] MSE / BCEWithLogits / CrossEntropy; AdamW
- [ ] training loop; validation loop; Dataset / DataLoader
- [ ] device-aware training; GPU OOM debugging; mixed precision
- [ ] gradient clipping; save/load `state_dict`
- [ ] Transformer shapes; Q/K/V; causal mask; LM loss shapes

> **Last check:** shape → dtype → device → target format → loss → finite values → gradients → updates → mode → learning rate.
